# 05 · Content-based filtering

A recommender can work from two kinds of information: what other users did (collaborative filtering, built in notebook 04), or what a movie is, such as its genres and release year (content-based filtering, built here). The content approach matters most for movies with few or no ratings, where there is too little user behaviour to learn from.

This notebook inspects the movie data, builds a content-based model, and tests it on rarely rated movies. Combining it with the collaborative model comes later, once the content model is fixed.

We use the same 17,241 movies as notebook 04. Movies dropped by the 5-rating filter are not in this catalogue. That filter counted all ratings, including the held-out ones, so this catalogue has almost no brand-new movies (0.02% of test ratings fall on movies with no training ratings). "Rare" below means few ratings, not newly released.

## Plan and rules (set before looking at results)
1. Inspect the data and fix the rare-movie slices. *Done.*
2. Build a genres + decade model with default settings. *Done, untuned.*
3. Tune it with a small ablation (genres only, decade only, a few decade weights, IDF on/off). Choose by the tail slice (view B), validation only. Then save the settings and validation table and tag the repo `content-v1`.
4. Try richer features (tags from users outside our sample, TMDB metadata). A new feature set is kept only if its paired 95% interval for the tail-slice view-B NDCG@10 difference against `content-v1` has a lower bound above zero. Otherwise we report "no measurable gain".
5. Retrain the collaborative model on train only (the saved one also saw validation, so it can't score validation), score it on the same slices, then blend and tune the blend weight on validation. Proposed rule, fixed before any blend result: maximise tail NDCG@10 in view A, as long as overall NDCG@10 stays at or above 98% of the collaborative model's validation score.
6. Run test once at the end, for all final candidates in one table.

Everything before step 6 is judged on validation only.

In [29]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "scripts"))   # notebooks/ -> project root -> scripts

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from recutils import load_split, to_sets

NAME, K, REL, SEED = "sample_10pct", 10, 4.0, 42

S = load_split(NAME)
train, val, test = S["train"], S["val"], S["test"]
n_users = int(S["user_map"].u.max()) + 1
n_items = int(S["item_map"].i.max()) + 1
print(f"{n_users:,} users | {n_items:,} movies in the modelled catalogue")

movies = pd.read_parquet(Path("../data/interim") / NAME / "movies.parquet")
meta   = S["item_map"].merge(movies, on="movieId", how="left").set_index("i").sort_index()
assert (meta.index.values == np.arange(n_items)).all() and meta.title.notna().all()


20,093 users | 17,241 movies in the modelled catalogue


## 1 · How much of the evaluation do rare movies make up?
We sort movies by how many users rated them 4 or higher in the training data ("positives"). A movie with few positives gives a collaborative model very little to learn from. A held-out rating of 4 or higher counts as "relevant", meaning a recommendation we'd want to get right.

The table shows, for each group, how many movies it contains and how many relevant validation and test interactions fall on it. This tells us whether rarely rated movies matter enough to the metrics for a content model to show any effect, and whether there are enough users among them to judge it fairly.

In [30]:
n_pos = np.bincount(train.i[train.rating >= REL], minlength=n_items)

bins   = [-1, 0, 4, 19, 99, 499, np.inf]
labels = ["0", "1-4", "5-19", "20-99", "100-499", "500+"]
bucket = pd.Series(pd.cut(n_pos, bins=bins, labels=labels), name="bucket")

def share_table(rel_df):
    b = pd.Series(bucket.values[rel_df.i.values])
    n_int = b.value_counts().reindex(labels)
    n_usr = pd.DataFrame({"u": rel_df.u.values, "b": b.values}).groupby("b", observed=False)["u"].nunique().reindex(labels)
    return n_int, n_usr

val_rel, test_rel = val[val.rating >= REL], test[test.rating >= REL]
v_int, v_usr = share_table(val_rel)
t_int, t_usr = share_table(test_rel)

summary = pd.DataFrame({
    "movies": bucket.value_counts().reindex(labels),
    "% of catalogue": (100 * bucket.value_counts().reindex(labels) / n_items).round(1),
    "relevant val interactions": v_int,
    "% of relevant val": (100 * v_int / v_int.sum()).round(2),
    "users (val)": v_usr,
    "relevant test interactions": t_int,
    "% of relevant test": (100 * t_int / t_int.sum()).round(2),
    "users (test)": t_usr,
})
summary

,movies,% of catalogue,relevant val interactions,% of relevant val,users (val),relevant test interactions,% of relevant test,users (test)
0,1095,6.4,80,0.11,77,121,0.16,116
1-4,5666,32.9,784,1.04,625,985,1.28,796
5-19,4836,28.0,2355,3.13,1761,2924,3.80,2220
20-99,3299,19.1,8460,11.23,5345,9646,12.54,6041
100-499,1685,9.8,21356,28.34,10663,22204,28.86,11236
500+,660,3.8,42318,56.16,14988,41049,53.36,15018


## 1b · How much metadata do rare movies have?
A content model can only use what's recorded about a movie. For each group, the table shows the share of movies with genres, the share with a release year, and the average number of genres per movie. If rare movies have little metadata, a content model can't help them. If movies have only about 2 genres each, many will look alike to it.

In [31]:
meta["year"]      = meta.title.str.extract(r"\((\d{4})\)\s*$")[0].astype("float")
meta["has_genre"] = meta.genres.ne("(no genres listed)")
meta["has_year"]  = meta.year.notna()
meta["n_genres"]  = meta.genres.where(meta.has_genre, "").str.count(r"\|").add(1).where(meta.has_genre, 0)
meta["bucket"]    = bucket.values

cov = meta.groupby("bucket", observed=False).agg(
    movies=("title", "size"),
    pct_with_genre=("has_genre", "mean"),
    pct_with_year=("has_year", "mean"),
    mean_n_genres=("n_genres", "mean"),
)
cov[["pct_with_genre", "pct_with_year"]] = (100 * cov[["pct_with_genre", "pct_with_year"]]).round(1)
cov["mean_n_genres"] = cov["mean_n_genres"].round(2)
cov.reindex(labels)

,movies,pct_with_genre,pct_with_year,mean_n_genres
bucket,,,,
0,1095,98.7,99.8,2.07
1-4,5666,98.9,99.7,1.98
5-19,4836,99.3,99.7,2.02
20-99,3299,99.8,99.7,2.22
100-499,1685,99.8,99.8,2.51
500+,660,100.0,100.0,2.92


**Observations**
- A small group of well-known movies carries most of the evaluation. The 660 movies with 500+ positives (3.8% of the catalogue) hold 56% of the relevant validation interactions. A positive is a training rating of 4 or higher; a relevant interaction is a held-out rating of 4 or higher.
- Movies with fewer than 20 positives are 67% of the catalogue but only 4.3% of the relevant validation interactions (5.2% on test). Movies with fewer than 5 positives are 39% of the catalogue and about 1.2% of the interactions (1.4% on test). The collaborative model has almost nothing to learn from for these.
- Almost every movie has metadata: at least 98.7% have genres and 99.7% have a year, in every group. But rare movies carry less of it, around 2 genres each against 2.9 for the most-rated group.

**What that might mean**
- Even a perfect model for rare movies could only touch about 4 to 5% of the relevant interactions. Overall NDCG will barely move when we add content features, so a change there wouldn't tell us much. The content model has to be judged on rare movies separately.
- Genres and year reach nearly every rare movie, but with about two genres each many movies will look alike to the model. We'll check how alike in the next step.

**Decisions (made now, before seeing any content-model results)**
- *Cold slice*: movies with fewer than 5 positives. *Tail slice*: fewer than 20. Tail is the main slice because it has enough users to measure; cold is reported next to it as the hardest case. Movies with zero positives (77 validation users) are too few to judge alone.
- Two ways to evaluate on a slice: (A) the normal top-10 over all movies, counting how many rare relevant movies show up, and (B) ranking only the slice's movies against each other, to see which model orders rare movies best.
- For test, slice membership is recomputed with train + validation, because that is the data the final model sees.
- Metrics are averaged over users with at least one relevant movie in the slice, with a 95% confidence interval from resampling users, since the slices are small.

## 1c · Fix the rare-movie slices
This cell turns the decisions above into code: flags marking which movies are in the cold and tail slices (for validation, using training data; for test, using train + validation), and a table of how big each slice is. "Random Recall@10" is what a random ranking would score if it had to pick from only that slice's movies. It's the floor for the in-slice evaluation.

In [32]:
def positives(df):
    return np.bincount(df.i[df.rating >= REL], minlength=n_items)

n_pos_val  = positives(train)                         # what the model knows when we tune on validation
n_pos_test = positives(pd.concat([train, val]))       # what the final model knows when we test

SLICES = {"cold": 5, "tail": 20}                      # fewer than this many positives
in_slice = {("val", s): n_pos_val < t for s, t in SLICES.items()}
in_slice.update({("test", s): n_pos_test < t for s, t in SLICES.items()})

def slice_table(rel_df, split):
    rows = {}
    for s, t in SLICES.items():
        movies_in = in_slice[(split, s)]
        hit = movies_in[rel_df.i.values]
        rows[f"{s} (<{t} positives)"] = {
            "movies": int(movies_in.sum()),
            "relevant interactions": int(hit.sum()),
            "% of all relevant": round(100 * hit.mean(), 2),
            "users with >=1": int(np.unique(rel_df.u.values[hit]).size),
            "random Recall@10": round(10 / movies_in.sum(), 4),
        }
    return pd.DataFrame(rows).T.astype({"movies": int, "relevant interactions": int, "users with >=1": int})

print("validation (movie counts from train)")
display(slice_table(val_rel, "val"))
print("test (movie counts from train + validation)")
display(slice_table(test_rel, "test"))

validation (movie counts from train)


,movies,relevant interactions,% of all relevant,users with >=1,random Recall@10
cold (<5 positives),6761,864,1.15,676,0.0015
tail (<20 positives),11597,3219,4.27,2144,0.0009


test (movie counts from train + validation)


,movies,relevant interactions,% of all relevant,users with >=1,random Recall@10
cold (<5 positives),6572,1033,1.34,814,0.0015
tail (<20 positives),11456,3802,4.94,2587,0.0009


## 2 · Turn movies into feature vectors
Genres become a multi-hot vector (one column per genre, 1 if the movie has it). Release decade becomes a one-hot vector. The next cells build both, weight the columns so rare features count more, and scale every row to length 1.

In [33]:
genres = meta.genres.str.get_dummies("|").drop(columns="(no genres listed)", errors="ignore").astype("int8")

decade  = (meta.year // 10 * 10).clip(lower=1910)           # anything before 1920 goes into one bucket
decades = pd.get_dummies(decade.map(lambda d: f"{int(d)}s" if pd.notna(d) else None)).astype("int8")

print("genre matrix:", genres.shape, "|", int(genres.sum(axis=1).eq(0).sum()), "movies with no genre")
print("decade matrix:", decades.shape, "|", int(decades.sum(axis=1).eq(0).sum()), "movies with no year")
genres.sum().sort_values(ascending=False)

genre matrix: (17241, 19) | 125 movies with no genre
decade matrix: (17241, 12) | 47 movies with no year


Drama          8082
Comedy         5819
Thriller       3284
Action         2676
Romance        2541
Crime          1959
Horror         1888
Adventure      1807
Sci-Fi         1495
Fantasy        1205
Children       1081
Animation      1068
Mystery        1050
Documentary     946
War             670
Musical         532
Western         325
IMAX            173
Film-Noir       164
dtype: int64

### Weighting and normalising
Every movie shares "Drama" with half the catalogue, so sharing it says little. Sharing "Film-Noir" says a lot. Each column is multiplied by its IDF, `log(N / movies that have the feature)`, so common features count less and rare ones count more. Then each row is scaled to length 1, so that a dot product between two movies is their cosine similarity.

In [34]:
B   = pd.concat([genres, decades], axis=1)                  # binary features, 17,241 x 31
idf = np.log(n_items / B.sum(axis=0))                       # rarer column -> larger weight

X    = B.values * idf.values                                # weight each column
norm = np.linalg.norm(X, axis=1, keepdims=True)
X    = np.divide(X, norm, out=np.zeros_like(X), where=norm > 0)   # rows of all zeros stay zero

print("feature matrix:", X.shape, "| movies with no features at all:", int((norm == 0).sum()))
idf.sort_values().round(2)

feature matrix: (17241, 31) | movies with no features at all: 15


Drama          0.76
Comedy         1.09
2010s          1.34
2000s          1.41
Thriller       1.66
1990s          1.79
Action         1.86
Romance        1.91
Crime          2.17
Horror         2.21
Adventure      2.26
1980s          2.30
Sci-Fi         2.45
Fantasy        2.66
Children       2.77
Animation      2.78
Mystery        2.80
Documentary    2.90
1970s          2.91
2020s          3.03
1960s          3.10
War            3.25
Musical        3.48
1950s          3.48
1940s          3.78
Western        3.97
1930s          4.23
IMAX           4.60
Film-Noir      4.66
1920s          5.11
1910s          5.84
dtype: float64

### How distinguishable are movies?
With only genres and decade, many movies end up with the exact same vector. A model can't rank identical vectors against each other, so this limits what it can do. This cell counts how many movies share their vector with at least one other movie, by rarity bucket.

In [35]:
vec_id = pd.util.hash_pandas_object(B, index=False)         # one hash per row; same row -> same hash
same   = vec_id.map(vec_id.value_counts()) - 1              # how many OTHER movies share this vector

print("distinct vectors:", vec_id.nunique(), "of", n_items, "movies")
print("sizes of the 5 biggest groups of identical movies:", vec_id.value_counts().head(5).tolist())

meta["shared"] = same > 0
meta.groupby("bucket", observed=False).agg(
    movies=("title", "size"),
    pct_sharing_a_vector=("shared", lambda s: round(100 * s.mean(), 1)),
)

distinct vectors: 3052 of 17241 movies
sizes of the 5 biggest groups of identical movies: [587, 531, 464, 391, 329]


,movies,pct_sharing_a_vector
bucket,,
0,1095,92.1
1-4,5666,90.9
5-19,4836,91.5
20-99,3299,89.1
100-499,1685,86.8
500+,660,81.2


### How many movies share each vector?
The 91% above counts a movie as sharing even if only one other movie has its vector. This cell counts how many *other* movies have exactly the same vector, and shows the median per rarity group and the share of movies with 10 or more identical others.

In [36]:
meta["identical_others"] = same        # movies with exactly the same vector as this one
meta.groupby("bucket", observed=False).agg(
    movies=("title", "size"),
    median_identical_others=("identical_others", "median"),
    pct_with_10_plus=("identical_others", lambda s: round(100 * (s >= 10).mean(), 1)),
)

,movies,median_identical_others,pct_with_10_plus
bucket,,,
0,1095,26.0,66.8
1-4,5666,29.0,69.3
5-19,4836,31.0,69.0
20-99,3299,24.0,64.8
100-499,1685,15.0,57.0
500+,660,8.0,45.2


## 3 · User profiles and content scores
A user's profile is the average feature vector of the movies they liked in train (rating ≥ 4). A movie's score for that user is the dot product of the profile with the movie's vector, so movies that look like what the user liked score highest. We hide movies the user already rated and take the top 10, using the same ranking and metric code as notebooks 03 and 04. First cell: imports and the "already seen" matrix.

In [37]:
import scipy.sparse as sp
from recutils import rank_topk, evaluate_topk

def seen_matrix(*dfs):
    """Sparse 0/1 matrix of (user, movie) pairs a user already has; used to hide them from recommendations."""
    d = pd.concat(dfs)
    return sp.csr_matrix((np.ones(len(d), dtype=np.float32), (d.u.values, d.i.values)),
                         shape=(n_users, n_items))

seen_train = seen_matrix(train)
rel_val = to_sets(val_rel)          # user -> set of validation movies they rated >= 4
print("validation users with >=1 relevant movie:", len(rel_val))

validation users with >=1 relevant movie: 18392


### Building the profiles
`L` marks which movies each user liked in train. Dividing each row by the user's number of liked movies and multiplying by `X` gives the average vector of their liked movies. Users with no liked movie get a profile of all zeros, so every movie scores 0 for them.

In [38]:
def build_profiles(df):
    liked = df[df.rating >= REL]
    L = sp.csr_matrix((np.ones(len(liked), dtype=np.float32), (liked.u.values, liked.i.values)),
                      shape=(n_users, n_items))                      # users x movies, 1 = liked
    n_liked = np.asarray(L.sum(axis=1)).ravel()
    inv = np.divide(1.0, n_liked, out=np.zeros_like(n_liked), where=n_liked > 0)
    return sp.diags(inv) @ L @ X, n_liked                            # mean of liked-movie vectors

profiles, n_liked = build_profiles(train)
print("profiles:", profiles.shape, "| users with no liked movie in train:", int((n_liked == 0).sum()))

profiles: (20093, 31) | users with no liked movie in train: 34


### Scoring and evaluating on validation
Scores come from `profile · movie vector`. Many movies have identical vectors, so many scores tie exactly. We add a tiny random number (about 1e-6) to every score so ties are broken randomly rather than by movie index. The seeded generator makes the run reproducible. Popularity is re-run next to it as the reference.

Three things to know about this setup:
- **Untuned.** Genre and decade columns have equal weight before IDF, and nothing here was chosen on validation. All content results in this notebook are default settings until the ablation.
- **Users with no liked movie in train (34)** have an all-zero profile, so they get random lists here. In notebook 04 they got the popularity list. The blend will use the popularity fallback for them.
- **Random tie-breaking** understates what this model could do. When many movies have the same vector, a real system would break ties with popularity or the collaborative score, which the blend will do.

In [39]:
rng = np.random.default_rng(SEED)
X32 = X.astype(np.float32)

def content_scores(ub, profiles):
    s = np.asarray(profiles[ub], dtype=np.float32) @ X32.T           # (batch, n_items)
    return s + 1e-6 * rng.random(s.shape, dtype=np.float32)          # random tie-break

ranked = rank_topk(lambda ub: content_scores(ub, profiles), rel_val.keys(), seen_train, K)
content_val = evaluate_topk(ranked, rel_val, n_items, K)

counts = np.bincount(train.i, minlength=n_items).astype(np.float32)
pop_val = evaluate_topk(rank_topk(lambda ub: np.tile(counts, (len(ub), 1)), rel_val.keys(), seen_train, K),
                        rel_val, n_items, K)

pd.DataFrame({"popularity": pop_val, "content (genres + decade)": content_val}).T.round(4)

,Recall@10,NDCG@10,HitRate@10,Coverage
popularity,0.0574,0.0415,0.1620,0.0137
content (genres + decade),0.0071,0.0047,0.0247,0.5449


### Looking at one user
A single user is an anecdote, but it shows what the model is doing. We print what the user liked, the top features of their profile, and the 10 movies recommended.

In [40]:
u = list(rel_val.keys())[0]
liked_u = train[(train.u == u) & (train.rating >= REL)].i
print("liked in train (first 8):", meta.loc[liked_u.head(8), "title"].tolist())

prof = profiles[u]
print("top profile features:", [(B.columns[j], round(float(prof[j]), 2)) for j in np.argsort(-prof)[:5]])
meta.loc[ranked[u], ["title", "genres"]]

liked in train (first 8): ['Being John Malkovich (1999)', 'Three Kings (1999)', 'Blade Runner (1982)', 'Emma (1996)', 'Trust (1990)', 'Shakespeare in Love (1998)', "Ferris Bueller's Day Off (1986)", 'Yojimbo (1961)']
top profile features: [('1990s', 0.25), ('1980s', 0.18), ('Drama', 0.16), ('Romance', 0.14), ('Comedy', 0.14)]


,title,genres
i,,
3648,Prelude to a Kiss (1992),Comedy|Drama|Romance
12575,"Same Love, Same Rain (El mismo amor, la misma ...",Comedy|Drama|Romance
7790,Venus Beauty Institute (Vénus beauté) (1999),Comedy|Drama|Romance
347,It Could Happen to You (1994),Comedy|Drama|Romance
1678,Hope Floats (1998),Comedy|Drama|Romance
1080,Delicatessen (1991),Comedy|Drama|Romance
2941,Singles (1992),Comedy|Drama|Romance
827,Twelfth Night (1996),Comedy|Drama|Romance
3473,"Couch in New York, A (1996)",Comedy|Drama|Romance


## 4 · Judging the content model on rare movies
Overall metrics barely reflect rare movies (they hold about 4% of relevant interactions), so we score the cold and tail slices separately, using validation only. Two views: (B) models rank only the slice's movies against each other, with random and popularity as baselines, and (A) the usual top 10 over all movies, counting only the rare relevant movies that show up. Each number comes with a 95% confidence interval from resampling users, because the slices are small. This first cell defines the helpers.

In [41]:
def per_user(ranked, rel):
    """Same metrics as evaluate_topk, but one value per user so we can bootstrap them."""
    disc = 1 / np.log2(np.arange(2, K + 2))
    rec, ndcg = [], []
    for u, r in rel.items():
        top = ranked[u][:K]
        g = np.array([m in r for m in top], dtype=float)
        rec.append(g.sum() / len(r))
        ndcg.append((g * disc[:len(top)]).sum() / disc[:min(len(r), K)].sum())
    return {"Recall@10": np.array(rec), "NDCG@10": np.array(ndcg)}

def slice_rel(rel, mask):
    """Keep only each user's relevant movies that fall in the slice; drop users left with none."""
    out = {u: {m for m in r if mask[m]} for u, r in rel.items()}
    return {u: r for u, r in out.items() if r}

def content_fn(ub):
    s = np.asarray(profiles[ub], dtype=np.float32) @ X32.T
    return s + 1e-6 * rng.random(s.shape, dtype=np.float32)

def pop_fn(ub):
    return np.tile(counts, (len(ub), 1)) + 1e-3 * rng.random((len(ub), n_items), dtype=np.float32)

def random_fn(ub):
    return rng.random((len(ub), n_items), dtype=np.float32)

MODELS = {"random": random_fn, "popularity": pop_fn, "content": content_fn}

### Running both views
`eval_slice` ranks for the users in a slice. With `restrict=True` every movie outside the slice gets a score of minus infinity, so only the slice's movies can be recommended (view B). With `restrict=False` all movies compete as normal (view A). `report` bootstraps over users and shows each mean with its 95% interval, plus the paired difference of content against the baselines.

Tie-breaking is random, so each model is run with 10 fixed seeds and the per-user scores are averaged before bootstrapping.

In [42]:
N_TIE_SEEDS = 10

def eval_slice(slice_name, restrict, names):
    global rng                                   # the model functions read this generator
    mask = in_slice[("val", slice_name)]
    rel = slice_rel(rel_val, mask)
    out = {}
    for name in names:
        fn = MODELS[name]
        f = (lambda ub, fn=fn: np.where(mask, fn(ub), -np.inf)) if restrict else fn
        runs = []
        for seed in range(N_TIE_SEEDS):
            rng = np.random.default_rng(seed)    # fresh, fixed generator per run
            runs.append(per_user(rank_topk(f, rel.keys(), seen_train, K), rel))
        out[name] = {met: np.mean([r[met] for r in runs], axis=0) for met in runs[0]}
    return out, len(rel)

def ci(vals, idx):
    m = vals[idx].mean(axis=1)
    return vals.mean(), *np.percentile(m, [2.5, 97.5])

def report(restrict, names):
    brng = np.random.default_rng(SEED)
    for s in SLICES:
        res, n = eval_slice(s, restrict, names)
        idx = brng.integers(0, n, (1000, n))                 # same resampled users for every model
        rows = {name: {met: "%.4f [%.4f, %.4f]" % ci(d[met], idx) for met in d} for name, d in res.items()}
        print(f"{s} slice, {n} users")
        display(pd.DataFrame(rows).T)
        for base in names:
            if base != "content":
                diff = res["content"]["NDCG@10"] - res[base]["NDCG@10"]
                print("  NDCG content - %s: %.4f [%.4f, %.4f]" % ((base,) + ci(diff, idx)))

### View B: ranking only within the slice
Which model orders rare movies best? Random is the floor.

In [43]:
report(restrict=True, names=["random", "popularity", "content"])

cold slice, 676 users


,Recall@10,NDCG@10
random,"0.0014 [0.0006, 0.0023]","0.0007 [0.0003, 0.0011]"
popularity,"0.0044 [0.0000, 0.0104]","0.0036 [0.0000, 0.0087]"
content,"0.0136 [0.0063, 0.0220]","0.0057 [0.0027, 0.0093]"


  NDCG content - random: 0.0051 [0.0021, 0.0086]
  NDCG content - popularity: 0.0021 [-0.0039, 0.0074]
tail slice, 2144 users


,Recall@10,NDCG@10
random,"0.0010 [0.0006, 0.0014]","0.0005 [0.0003, 0.0007]"
popularity,"0.0041 [0.0019, 0.0069]","0.0018 [0.0008, 0.0031]"
content,"0.0104 [0.0072, 0.0139]","0.0053 [0.0036, 0.0072]"


  NDCG content - random: 0.0048 [0.0030, 0.0067]
  NDCG content - popularity: 0.0035 [0.0012, 0.0057]


### View A: top 10 over all movies, counting only rare relevant movies
Popularity can't recommend rare movies by design, so it should score 0 here. This shows whether content features surface rare movies at all when competing with popular ones.

In [44]:
report(restrict=False, names=["popularity", "content"])

cold slice, 676 users


,Recall@10,NDCG@10
popularity,"0.0000 [0.0000, 0.0000]","0.0000 [0.0000, 0.0000]"
content,"0.0046 [0.0004, 0.0099]","0.0016 [0.0002, 0.0035]"


  NDCG content - popularity: 0.0016 [0.0002, 0.0035]
tail slice, 2144 users


,Recall@10,NDCG@10
popularity,"0.0000 [0.0000, 0.0000]","0.0000 [0.0000, 0.0000]"
content,"0.0067 [0.0040, 0.0099]","0.0031 [0.0020, 0.0045]"


  NDCG content - popularity: 0.0031 [0.0020, 0.0045]


## Observations (steps 3 and 4)
All results here use default settings (genre and decade columns weighted equally, nothing tuned). The ablation comes next.

- Genres and decade give 3,052 distinct vectors for 17,241 movies, and about 91% of rare movies share a vector with another movie. That number doesn't say whether a movie shares with 1 other or with 500; the group-size cell above shows that. Ties are broken randomly, and every slice result averages 10 tie-break seeds.
- Over all movies, content alone is far below popularity (validation NDCG@10 0.005 vs 0.042) but recommends a much wider range of movies (coverage 0.54 vs 0.014).
- Within the tail slice (<20 positives, 2,144 users), content gets NDCG@10 0.0053 against 0.0018 for popularity and 0.0005 for random. Content minus popularity is +0.0035 [0.0012, 0.0057].
- Within the cold slice (<5 positives, 676 users), content gets NDCG@10 0.0057 [0.0027, 0.0093] and popularity 0.0036 [0.0000, 0.0087]; random gets 0.0007 [0.0003, 0.0011]. Content is clearly above random. Popularity's interval reaches zero, so we can't say the same for it. Content minus popularity is +0.0021 [-0.0039, 0.0074], so this sample can't separate them.
- When competing against all movies, popularity never recommends a rare movie. Content gets NDCG@10 0.0016 (cold) and 0.0031 (tail): small, but above zero.
- Recall@10 within the tail slice is about 1% (random: 0.1%), so only about 1 in 100 of a user's relevant rare movies reaches their top 10. A likely reason is that many movies share an identical vector (the example user's top 10 are all the same Comedy|Drama|Romance kind of movie), but we haven't tested that.
- The example user's profile ranks decades (1990s 0.25, 1980s 0.18) above genres (Drama 0.16), so decade may be overweighted. That is what the ablation checks.
- A single tie-break seed moved the cold numbers noticeably between runs, so single-run results on small slices shouldn't be trusted.

**Next:** the ablation on the tail slice (validation only), then save the chosen settings and tag `content-v1`. Tags and TMDB come after that, and the blend after those (see the plan at the top).

## 5 · Feature ablation: which features help rare-movie recommendations?

The current content-based model uses movie genres and release decade, with IDF weighting. Before adding richer metadata, we need to understand how much each feature contributes.

We will compare genres only, decade only, and combinations with different decade weights. Each configuration will be tested with IDF weighting enabled and disabled.

### Evaluation protocol

- Use the existing training and validation split. Do not use test results to choose settings.
- Select the configuration using tail-slice NDCG@10 in View B, where models rank movies only against other movies in the tail slice.
- Report cold-slice results alongside the tail results.
- Keep the configuration grid fixed before running the experiment.
- Save the validation results separately from the final model configuration.

The goal is to understand which choices improve rare-movie recommendations and whether the observed differences are meaningful.

### Build a feature matrix with selectable weights

This function starts from the existing binary feature table `B`: genre columns first, decade columns second. The two multipliers can turn a feature group off or reduce its contribution. IDF optionally gives rarer features more weight. Finally, each movie vector is normalized so its length does not dominate its similarity score.

In [45]:
def make_feature_matrix(
    genre_weight=1.0,
    decade_weight=1.0,
    use_idf=True,
):
    # B already contains genres followed by decade features.
    binary_features = B.to_numpy(dtype=np.float32)

    # Begin with equal weights for every feature.
    weights = np.ones(B.shape[1], dtype=np.float32)

    # The first columns represent genres.
    n_genre_features = genres.shape[1]
    weights[:n_genre_features] *= genre_weight

    # The remaining columns represent decades.
    weights[n_genre_features:] *= decade_weight

    # Optionally downweight common features.
    if use_idf:
        feature_counts = B.sum(axis=0).to_numpy(dtype=np.float32)
        idf_weights = np.log(
            n_items / np.maximum(feature_counts, 1.0)
        )
        weights *= idf_weights

    # Apply the selected weights to every movie.
    X_features = binary_features * weights

    # Normalize each movie vector to unit length.
    row_norms = np.linalg.norm(
        X_features, axis=1, keepdims=True
    )
    X_features = np.divide(
        X_features,
        row_norms,
        out=np.zeros_like(X_features),
        where=row_norms > 0,
    )

    return X_features

### Verify the default features are unchanged

Before testing alternatives, confirm that weights of 1.0 for both groups with IDF enabled reproduce the original matrix `X`. The small numerical tolerance allows for harmless floating-point differences. If this assertion fails, stop and fix it before running the experiment.

In [46]:
X_default_check = make_feature_matrix(
    genre_weight=1.0,
    decade_weight=1.0,
    use_idf=True,
)

max_difference = float(
    np.max(np.abs(X_default_check - X))
)

print("Maximum absolute difference:", max_difference)

assert np.allclose(
    X_default_check,
    X,
    atol=1e-6,
), "The new feature builder does not match the existing model."

Maximum absolute difference: 1.0267299244137718e-07


### Build profiles for each feature configuration

Each feature matrix needs its own user profiles. We average the feature vectors of movies the user rated 4 or higher in the training split. Passing the matrix into the function prevents us from accidentally reusing profiles built with a different feature set.

In [47]:
def build_profiles_for(df, X_features):
    # Keep positive ratings from the supplied dataset.
    liked = df[df.rating >= REL]

    # Construct the user-movie interaction matrix.
    L = sp.csr_matrix(
        (
            np.ones(len(liked), dtype=np.float32),
            (liked.u.values, liked.i.values),
        ),
        shape=(n_users, n_items),
    )

    # Count how many liked movies each user has.
    n_liked = np.asarray(L.sum(axis=1)).ravel()

    # Calculate 1 / number of liked movies.
    inv = np.divide(
        1.0,
        n_liked,
        out=np.zeros_like(n_liked),
        where=n_liked > 0,
    )

    # Average the feature vectors of each user's liked movies.
    profiles = sp.diags(inv) @ L @ X_features

    return np.asarray(profiles, dtype=np.float32), n_liked

### Check the profile builder

With the default features, these new profiles should match the profiles already used by the original content model. This sanity check catches implementation differences before we compare alternative configurations.

In [48]:
profiles_default_check, n_liked_check = build_profiles_for(
    train,
    X_default_check,
)

print("Profiles shape:", profiles_default_check.shape)
print("Original profiles shape:", profiles.shape)

assert profiles_default_check.shape == profiles.shape

assert np.allclose(
    profiles_default_check,
    profiles,
    atol=1e-5,
), "The new profiles do not match the existing profiles."

Profiles shape: (20093, 31)
Original profiles shape: (20093, 31)


### Create a content scorer

The scorer uses the feature matrix and profiles from the same configuration. Because movie vectors are normalized, the dot product acts like a cosine-similarity score: higher means a better match to the user's profile. The tiny random term breaks ties without materially changing genuine score differences.

In [49]:
def make_content_scorer(X_features, user_profiles):
    item_features = np.asarray(
        X_features, dtype=np.float32
    )
    user_profiles = np.asarray(
        user_profiles, dtype=np.float32
    )

    def score(user_batch):
        scores = (
            user_profiles[user_batch] @ item_features.T
        )

        # Break exact or near-identical-score ties reproducibly.
        return scores + 1e-6 * rng.random(
            scores.shape, dtype=np.float32
        )

    return score

### Smoke-test the scorer

Test the default scorer on three users before running the grid. The returned matrix should have one row per user and one column per movie, with only finite scores. Users with no liked positive movie have zero profiles, so this content-only scorer effectively gives them random rankings; we should use one consistent fallback policy when we later build the hybrid.

In [50]:
default_scorer = make_content_scorer(
    X_default_check,
    profiles_default_check,
)

sample_users = np.asarray(
    list(rel_val.keys())[:3],
    dtype=np.int32,
)

rng = np.random.default_rng(SEED)
sample_scores = default_scorer(sample_users)

print("Score matrix shape:", sample_scores.shape)

assert sample_scores.shape == (3, n_items)
assert np.isfinite(sample_scores).all()

Score matrix shape: (3, 17241)


### Fix the 10 configurations before evaluating

We compare genres only, decade only, and three genre-plus-decade weightings. Each variant is run with IDF weighting both off and on, giving 10 configurations. `DEFAULT_CONFIG` names the original model so that every alternative can be compared against the same reference.

In [51]:
FEATURE_VARIANTS = [
    ("genres_only", 1.0, 0.0),
    ("decade_only", 0.0, 1.0),
    ("both_decade_0.25", 1.0, 0.25),
    ("both_decade_0.5", 1.0, 0.5),
    ("both_decade_1.0", 1.0, 1.0),
]

CONFIGS = []

for variant, genre_weight, decade_weight in FEATURE_VARIANTS:
    for use_idf in (False, True):
        CONFIGS.append({
            "name": (
                f"{variant}_idf_"
                f"{'on' if use_idf else 'off'}"
            ),
            "genre_weight": genre_weight,
            "decade_weight": decade_weight,
            "use_idf": use_idf,
        })

config_table = pd.DataFrame(CONFIGS)

display(config_table)

assert len(CONFIGS) == 10

DEFAULT_CONFIG = "both_decade_1.0_idf_on"

assert sum(
    cfg["name"] == DEFAULT_CONFIG for cfg in CONFIGS
) == 1

,name,genre_weight,decade_weight,use_idf
0,genres_only_idf_off,1.0,0.00,False
1,genres_only_idf_on,1.0,0.00,True
2,decade_only_idf_off,0.0,1.00,False
3,decade_only_idf_on,0.0,1.00,True
4,both_decade_0.25_idf_off,1.0,0.25,False
5,both_decade_0.25_idf_on,1.0,0.25,True
6,both_decade_0.5_idf_off,1.0,0.50,False
7,both_decade_0.5_idf_on,1.0,0.50,True
8,both_decade_1.0_idf_off,1.0,1.00,False
9,both_decade_1.0_idf_on,1.0,1.00,True


### Define the evaluation procedure

For each slice, we keep only users who have at least one relevant validation movie in that slice. In View B, movies outside the selected slice are masked so they cannot enter the top 10. Each configuration uses the same 10 tie-breaking seeds, and we average each user's metrics over those runs before calculating confidence intervals.

In [52]:
def eval_content_configuration(slice_name, scorer):
    global rng

    # Always use validation slice membership.
    mask = in_slice[("val", slice_name)]

    # Keep users who have relevant validation movies in this slice.
    rel = slice_rel(rel_val, mask)

    runs = []

    for seed in range(N_TIE_SEEDS):
        # Use the same fixed seeds for every configuration.
        rng = np.random.default_rng(seed)

        def restricted_scores(user_batch):
            scores = scorer(user_batch)

            # View B: only movies belonging to this slice compete.
            return np.where(mask, scores, -np.inf)

        ranked = rank_topk(
            restricted_scores,
            rel.keys(),
            seen_train,
            K,
        )

        runs.append(per_user(ranked, rel))

    # Average each user's metric over tie-breaking seeds.
    averaged = {
        metric: np.mean(
            [run[metric] for run in runs],
            axis=0,
        )
        for metric in runs[0]
    }

    return averaged, len(rel)

### Run the fixed configuration grid

For every configuration, rebuild the item features, rebuild profiles from **training data only**, and evaluate on validation. Tail-slice NDCG@10 in View B is the primary selection metric; cold-slice results are reported as a secondary diagnostic. This cell runs all 10 pre-specified configurations without looking at test results.

In [53]:
ablation_runs = {}

for number, cfg in enumerate(CONFIGS, start=1):
    # Build the item features for this configuration.
    X_cfg = make_feature_matrix(
        genre_weight=cfg["genre_weight"],
        decade_weight=cfg["decade_weight"],
        use_idf=cfg["use_idf"],
    )

    # Build user profiles from training data only.
    profiles_cfg, n_liked_cfg = build_profiles_for(
        train,
        X_cfg,
    )

    # Create the scorer that uses these exact features/profiles.
    scorer = make_content_scorer(
        X_cfg,
        profiles_cfg,
    )

    # Primary evaluation: tail slice, View B.
    tail_metrics, n_tail = eval_content_configuration(
        "tail",
        scorer,
    )

    # Secondary evaluation: cold slice, View B.
    cold_metrics, n_cold = eval_content_configuration(
        "cold",
        scorer,
    )

    ablation_runs[cfg["name"]] = {
        "tail": tail_metrics,
        "cold": cold_metrics,
        "n_tail_users": n_tail,
        "n_cold_users": n_cold,
    }

    print(
        f"{number}/{len(CONFIGS)} — {cfg['name']} | "
        f"tail NDCG@10: "
        f"{tail_metrics['NDCG@10'].mean():.5f} | "
        f"cold NDCG@10: "
        f"{cold_metrics['NDCG@10'].mean():.5f}"
    )

1/10 — genres_only_idf_off | tail NDCG@10: 0.00263 | cold NDCG@10: 0.00452
2/10 — genres_only_idf_on | tail NDCG@10: 0.00335 | cold NDCG@10: 0.00501
3/10 — decade_only_idf_off | tail NDCG@10: 0.00101 | cold NDCG@10: 0.00164
4/10 — decade_only_idf_on | tail NDCG@10: 0.00101 | cold NDCG@10: 0.00164
5/10 — both_decade_0.25_idf_off | tail NDCG@10: 0.00475 | cold NDCG@10: 0.00425
6/10 — both_decade_0.25_idf_on | tail NDCG@10: 0.00531 | cold NDCG@10: 0.00609
7/10 — both_decade_0.5_idf_off | tail NDCG@10: 0.00549 | cold NDCG@10: 0.00471
8/10 — both_decade_0.5_idf_on | tail NDCG@10: 0.00541 | cold NDCG@10: 0.00449
9/10 — both_decade_1.0_idf_off | tail NDCG@10: 0.00584 | cold NDCG@10: 0.00565
10/10 — both_decade_1.0_idf_on | tail NDCG@10: 0.00529 | cold NDCG@10: 0.00563


### Estimate uncertainty with paired user-level bootstrap intervals

A small NDCG difference can occur by chance because the evaluation users are only one sample of possible users. We resample users with replacement 1,000 times. The **same sampled users are used for every configuration**, so differences versus the default are paired comparisons. Tail and cold slices use separate samples because they contain different sets of users.

In [54]:
# The default configuration is the reference for paired comparisons.
default_tail = ablation_runs[DEFAULT_CONFIG]["tail"]["NDCG@10"]
default_cold = ablation_runs[DEFAULT_CONFIG]["cold"]["NDCG@10"]

# Check that every configuration was evaluated on the same number of users.
tail_lengths = {len(ablation_runs[cfg["name"]]["tail"]["NDCG@10"]) for cfg in CONFIGS}
cold_lengths = {len(ablation_runs[cfg["name"]]["cold"]["NDCG@10"]) for cfg in CONFIGS}
assert len(tail_lengths) == 1 and len(cold_lengths) == 1

n_tail_users = len(default_tail)
n_cold_users = len(default_cold)
bootstrap_rng = np.random.default_rng(SEED)
N_BOOTSTRAP = 1000

# Each row contains the user indices for one bootstrap resample.
tail_idx = bootstrap_rng.integers(0, n_tail_users, size=(N_BOOTSTRAP, n_tail_users))
cold_idx = bootstrap_rng.integers(0, n_cold_users, size=(N_BOOTSTRAP, n_cold_users))

print(f"Tail slice: {n_tail_users:,} users | Cold slice: {n_cold_users:,} users")
print(f"Bootstrap resamples: {N_BOOTSTRAP:,}")

Tail slice: 2,144 users | Cold slice: 676 users
Bootstrap resamples: 1,000


### Compare validation performance and paired differences

For every configuration, we report tail NDCG@10 with a 95% bootstrap interval, the paired NDCG difference versus the original configuration, and cold-slice NDCG with its interval. Recall is included as a secondary metric. We sort by the **tail NDCG@10 point estimate**, as pre-specified; the paired difference interval tells us whether the apparent gain over the default is clear or uncertain.

In [55]:
rows = []

for cfg in CONFIGS:
    name = cfg["name"]
    tail = ablation_runs[name]["tail"]
    cold = ablation_runs[name]["cold"]

    tail_ndcg = tail["NDCG@10"]
    cold_ndcg = cold["NDCG@10"]

    tail_mean, tail_low, tail_high = ci(tail_ndcg, tail_idx)
    cold_mean, cold_low, cold_high = ci(cold_ndcg, cold_idx)

    # Subtract user by user, then bootstrap these paired differences.
    tail_delta = tail_ndcg - default_tail
    delta_mean, delta_low, delta_high = ci(tail_delta, tail_idx)

    rows.append({
        "config": name,
        "genre_weight": cfg["genre_weight"],
        "decade_weight": cfg["decade_weight"],
        "use_idf": cfg["use_idf"],
        "tail_NDCG@10": tail_mean,
        "tail_NDCG_CI_low": tail_low,
        "tail_NDCG_CI_high": tail_high,
        "tail_delta_vs_default": delta_mean,
        "delta_CI_low": delta_low,
        "delta_CI_high": delta_high,
        "tail_Recall@10": tail["Recall@10"].mean(),
        "cold_NDCG@10": cold_mean,
        "cold_NDCG_CI_low": cold_low,
        "cold_NDCG_CI_high": cold_high,
        "cold_Recall@10": cold["Recall@10"].mean(),
    })

results_df = (
    pd.DataFrame(rows)
    .sort_values("tail_NDCG@10", ascending=False)
    .reset_index(drop=True)
)

# A compact display is easier to compare in the notebook; the CSV keeps
# the numerical lower and upper bounds in separate columns.
display_df = pd.DataFrame({
    "Configuration": results_df["config"],
    "Tail NDCG@10": results_df["tail_NDCG@10"].map(lambda x: f"{x:.5f}"),
    "Tail NDCG 95% CI": results_df.apply(lambda r: f"[{r['tail_NDCG_CI_low']:.5f}, {r['tail_NDCG_CI_high']:.5f}]", axis=1),
    "Delta vs default": results_df["tail_delta_vs_default"].map(lambda x: f"{x:+.5f}"),
    "Paired delta 95% CI": results_df.apply(lambda r: f"[{r['delta_CI_low']:.5f}, {r['delta_CI_high']:.5f}]", axis=1),
    "Tail Recall@10": results_df["tail_Recall@10"].map(lambda x: f"{x:.5f}"),
    "Cold NDCG@10": results_df["cold_NDCG@10"].map(lambda x: f"{x:.5f}"),
    "Cold NDCG 95% CI": results_df.apply(lambda r: f"[{r['cold_NDCG_CI_low']:.5f}, {r['cold_NDCG_CI_high']:.5f}]", axis=1),
    "Cold Recall@10": results_df["cold_Recall@10"].map(lambda x: f"{x:.5f}"),
})
display(display_df)

,Configuration,Tail NDCG@10,Tail NDCG 95% CI,Delta vs default,Paired delta 95% CI,Tail Recall@10,Cold NDCG@10,Cold NDCG 95% CI,Cold Recall@10
0,both_decade_1.0_idf_off,0.00584,"[0.00390, 0.00795]",+0.00055,"[-0.00087, 0.00210]",0.01153,0.00565,"[0.00236, 0.00943]",0.01114
1,both_decade_0.5_idf_off,0.00549,"[0.00365, 0.00765]",+0.00021,"[-0.00143, 0.00195]",0.01043,0.00471,"[0.00185, 0.00774]",0.01053
2,both_decade_0.5_idf_on,0.00541,"[0.00364, 0.00746]",+0.00013,"[-0.00099, 0.00128]",0.01067,0.00449,"[0.00180, 0.00744]",0.01001
3,both_decade_0.25_idf_on,0.00531,"[0.00329, 0.00776]",+0.00002,"[-0.00182, 0.00210]",0.00918,0.00609,"[0.00264, 0.01043]",0.01240
4,both_decade_1.0_idf_on,0.00529,"[0.00353, 0.00726]",+0.00000,"[0.00000, 0.00000]",0.01046,0.00563,"[0.00252, 0.00902]",0.01349
5,both_decade_0.25_idf_off,0.00475,"[0.00302, 0.00680]",-0.00053,"[-0.00228, 0.00119]",0.00836,0.00425,"[0.00146, 0.00735]",0.00962
6,genres_only_idf_on,0.00335,"[0.00182, 0.00526]",-0.00194,"[-0.00387, 0.00017]",0.00581,0.00501,"[0.00179, 0.00896]",0.01044
7,genres_only_idf_off,0.00263,"[0.00132, 0.00433]",-0.00265,"[-0.00467, -0.00063]",0.00379,0.00452,"[0.00123, 0.00846]",0.00810
8,decade_only_idf_off,0.00101,"[0.00075, 0.00128]",-0.00428,"[-0.00626, -0.00250]",0.00212,0.00164,"[0.00099, 0.00240]",0.00328
9,decade_only_idf_on,0.00101,"[0.00075, 0.00128]",-0.00428,"[-0.00626, -0.00250]",0.00212,0.00164,"[0.00099, 0.00240]",0.00328


### Read the table before choosing a configuration

- The first row is the **numerical leader** for tail-slice NDCG@10; that alone does not prove it is a reliable improvement.
- A paired 95% interval for `tail_delta_vs_default` entirely above zero supports an improvement over the existing model on this validation slice. If the interval includes zero, describe the result as uncertain rather than a confirmed gain.
- Check cold performance too. It is a secondary diagnostic, not the metric used to select the winner.
- With decade-only features, IDF on/off may produce identical scores: a dated movie has only one active decade feature, and row normalization can cancel that feature's scalar weight. That is a consequence of the representation, not necessarily a coding error.
- The original default should reproduce approximately 0.0053 tail NDCG@10 and 0.0057 cold NDCG@10. If it differs materially, investigate before accepting the experiment.

In [56]:
best = results_df.iloc[0]
default_row = results_df.loc[results_df["config"] == DEFAULT_CONFIG].iloc[0]

print("Numerical leader:", best["config"])
print(f"Tail NDCG@10: {best['tail_NDCG@10']:.5f}")
print(
    "Paired difference versus default: "
    f"{best['tail_delta_vs_default']:+.5f} "
    f"[{best['delta_CI_low']:+.5f}, {best['delta_CI_high']:+.5f}]"
)
print(
    "Cold NDCG@10: "
    f"{best['cold_NDCG@10']:.5f} "
    f"[{best['cold_NDCG_CI_low']:.5f}, {best['cold_NDCG_CI_high']:.5f}]"
)
print(
    "Default configuration — tail / cold NDCG@10: "
    f"{default_row['tail_NDCG@10']:.5f} / {default_row['cold_NDCG@10']:.5f}"
)

if best["config"] == DEFAULT_CONFIG:
    print("The original configuration is the numerical leader.")
elif best["delta_CI_low"] > 0:
    print("The leader's paired 95% interval is above zero: evidence of a validation gain over the default.")
else:
    print("The leader has the highest point estimate, but the paired interval does not establish a clear gain over the default.")

print("No model configuration has been frozen by this cell.")

Numerical leader: both_decade_1.0_idf_off
Tail NDCG@10: 0.00584
Paired difference versus default: +0.00055 [-0.00087, +0.00210]
Cold NDCG@10: 0.00565 [0.00236, 0.00943]
Default configuration — tail / cold NDCG@10: 0.00529 / 0.00563
The leader has the highest point estimate, but the paired interval does not establish a clear gain over the default.
No model configuration has been frozen by this cell.


### Save the preliminary ablation report

Save the full numerical table so the results can be inspected, compared, and reproduced. This is an **ablation report**, not the official `content-v1` configuration. Do not create `content_v1_config.json`, commit a freeze, or run the test evaluation yet; review the table and paired intervals first.

In [57]:
reports_dir = Path("../reports")
reports_dir.mkdir(parents=True, exist_ok=True)

report_path = reports_dir / "content_ablation_validation.csv"
results_df.to_csv(report_path, index=False)
print("Saved preliminary validation results to:", report_path)

Saved preliminary validation results to: ..\reports\content_ablation_validation.csv


## 6 · Review the ablation results

The ten feature configurations have been evaluated on the validation set.
We select the configuration with the highest tail-slice NDCG@10 in View B,
while also examining its paired confidence interval and cold-slice performance.

The winning configuration is a candidate for our frozen content baseline.
A higher point estimate alone does not prove a reliable improvement.

In [58]:
ablation_path = Path("../reports/content_ablation_validation.csv")

assert ablation_path.exists(), "Ablation report not found."

ablation_check = pd.read_csv(ablation_path)

assert len(ablation_check) == 10, "Expected 10 configurations."
assert ablation_check["config"].is_unique

display(
    ablation_check[
        [
            "config",
            "tail_NDCG@10",
            "tail_delta_vs_default",
            "delta_CI_low",
            "delta_CI_high",
            "cold_NDCG@10",
        ]
    ]
)

,config,tail_NDCG@10,tail_delta_vs_default,delta_CI_low,delta_CI_high,cold_NDCG@10
0,both_decade_1.0_idf_off,0.005840,0.000553,-0.000873,0.002099,0.005647
1,both_decade_0.5_idf_off,0.005494,0.000206,-0.001432,0.001951,0.004714
2,both_decade_0.5_idf_on,0.005413,0.000125,-0.000989,0.001278,0.004494
3,both_decade_0.25_idf_on,0.005306,0.000018,-0.001818,0.002104,0.006085
4,both_decade_1.0_idf_on,0.005287,0.000000,0.000000,0.000000,0.005630
5,both_decade_0.25_idf_off,0.004754,-0.000533,-0.002285,0.001186,0.004250
6,genres_only_idf_on,0.003348,-0.001939,-0.003866,0.000170,0.005010
7,genres_only_idf_off,0.002634,-0.002653,-0.004667,-0.000635,0.004517
8,decade_only_idf_off,0.001009,-0.004278,-0.006260,-0.002504,0.001643
9,decade_only_idf_on,0.001009,-0.004278,-0.006260,-0.002504,0.001643


### 6.1 · Record the frozen model's validation performance

Now that the feature configuration has been selected, we evaluate it
using the same validation protocol as the original content model.

We compare content-v1 with random and popularity baselines in both
views, for the cold and tail slices. Paired confidence intervals show
the uncertainty in the differences between models.

This table is the official validation record for content-v1.
The test set remains untouched.

In [59]:
X_content_v1 = make_feature_matrix(
    genre_weight=1.0,
    decade_weight=1.0,
    use_idf=False,
)

profiles_content_v1, n_liked_content_v1 = build_profiles_for(
    train,
    X_content_v1,
)

content_v1_scorer = make_content_scorer(
    X_content_v1,
    profiles_content_v1,
)

# From this point on, the content entry refers to the selected v1.
MODELS["content"] = content_v1_scorer

print("Content-v1 feature matrix:", X_content_v1.shape)
print("Content-v1 user profiles:", profiles_content_v1.shape)

assert X_content_v1.shape == X.shape
assert profiles_content_v1.shape == profiles.shape
assert np.isfinite(X_content_v1).all()

Content-v1 feature matrix: (17241, 31)
Content-v1 user profiles: (20093, 31)


In [60]:
official_rows = []

N_BOOTSTRAP_OFFICIAL = 1000
bootstrap_rng = np.random.default_rng(SEED)
bootstrap_indices = {}

for view, restrict in [("B", True), ("A", False)]:
    for slice_name in ("cold", "tail"):

        evaluated, n_eval_users = eval_slice(
            slice_name,
            restrict=restrict,
            names=["random", "popularity", "content"],
        )

        # Use the same bootstrap samples for all models
        # and both views of the same rarity slice.
        if slice_name not in bootstrap_indices:
            bootstrap_indices[slice_name] = bootstrap_rng.integers(
                0,
                n_eval_users,
                size=(N_BOOTSTRAP_OFFICIAL, n_eval_users),
            )

        idx = bootstrap_indices[slice_name]

        for model_name, metrics in evaluated.items():
            model_label = (
                "content-v1" if model_name == "content"
                else model_name
            )

            for metric_name in ("NDCG@10", "Recall@10"):
                mean, low, high = ci(metrics[metric_name], idx)

                official_rows.append({
                    "model": model_label,
                    "comparison": "model_metric",
                    "view": view,
                    "slice": slice_name,
                    "n_users": n_eval_users,
                    "metric": metric_name,
                    "estimate": float(mean),
                    "ci95_low": float(low),
                    "ci95_high": float(high),
                })

        # Paired NDCG differences: content-v1 minus each baseline.
        for baseline_name in ("random", "popularity"):
            delta = (
                evaluated["content"]["NDCG@10"]
                - evaluated[baseline_name]["NDCG@10"]
            )

            mean, low, high = ci(delta, idx)

            official_rows.append({
                "model": "content-v1",
                "comparison": f"content-v1 minus {baseline_name}",
                "view": view,
                "slice": slice_name,
                "n_users": n_eval_users,
                "metric": "NDCG@10 difference",
                "estimate": float(mean),
                "ci95_low": float(low),
                "ci95_high": float(high),
            })

content_v1_val = pd.DataFrame(official_rows)

reports_dir = Path("../reports")
reports_dir.mkdir(parents=True, exist_ok=True)

content_v1_val.to_csv(
    reports_dir / "content_v1_val.csv",
    index=False,
)

display(content_v1_val.round(6))
print("Official content-v1 validation report saved.")

,model,comparison,view,slice,n_users,metric,estimate,ci95_low,ci95_high
0,random,model_metric,B,cold,676,NDCG@10,0.000661,0.000257,0.001104
1,random,model_metric,B,cold,676,Recall@10,0.001405,0.000592,0.002293
2,popularity,model_metric,B,cold,676,NDCG@10,0.003596,0.000000,0.008671
3,popularity,model_metric,B,cold,676,Recall@10,0.004438,0.000000,0.010355
4,content-v1,model_metric,B,cold,676,NDCG@10,0.005647,0.002224,0.009882
5,content-v1,model_metric,B,cold,676,Recall@10,0.011144,0.004585,0.018984
6,content-v1,content-v1 minus random,B,cold,676,NDCG@10 difference,0.004986,0.001629,0.009293
7,content-v1,content-v1 minus popularity,B,cold,676,NDCG@10 difference,0.002051,-0.004280,0.007759
8,random,model_metric,B,tail,2144,NDCG@10,0.000514,0.000313,0.000747
9,random,model_metric,B,tail,2144,Recall@10,0.001000,0.000619,0.001436


Official content-v1 validation report saved.


The feature ablation is complete. We selected genres and decade
features with equal multipliers and IDF weighting disabled, based on
validation tail-slice NDCG@10 in View B.

This configuration is now the fixed reference for evaluating richer
content features. Its improvement over the previous default is
inconclusive, so we do not claim a proven performance gain.

The selected settings and validation results are saved under reports/.
No test-set evaluation was performed during this selection.